# CellViz tutorial

CellViz plays back the recordings of one field of your **populated** schema with the ROI mask of each presentation,
and shows the traces of the ROIs in sync with the movie and the stimulus triggers, next to their trial mean ± SD.

It is **read-only**: it only reads your tables (and the recording files referenced by `Presentation`), never inserts,
updates or deletes anything, and adds no table to your schema. MP4 exports are written to your `processed` store
(folder `cellviz/`), never into the raw data.

What you need:
- a schema in which at least `RoiMask` and `Traces` are populated. Optional, if they are in your schema and
  populated: `PreprocessTraces` (preprocessed traces), `ChirpQI` / `OsDsIndexes` (quality filter), `HighRes`
  (HighRes view), `Snippets` (repeat settings for the trial mean),
- the optional dependencies: `uv pip install -e ".[cellviz]"` (ipywidgets, imageio-ffmpeg for the MP4 export,
  ipympl for clicking ROIs in the figure).

This notebook
1. connects to your schema,
2. lists the fields CellViz can show and opens one,
3. shows how to drive the viewer from code (overlays, views, selection, quality filter),
4. exports an MP4 into your `processed` store.

Before running it, replace the placeholders in section 1 (path of your DataJoint config file, `YOUR_PROJECT`).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Clickable figure (click ROIs in the field view or heatmap, click the traces to seek) needs ipympl.
# Without it the figure is shown as an image and ROIs are selected in the list next to it.
USE_WIDGET_BACKEND = True

if USE_WIDGET_BACKEND:
    try:
        get_ipython().run_line_magic('matplotlib', 'widget')
    except Exception as e:  # ipympl not installed
        print(f"%matplotlib widget not available ({type(e).__name__}: {e}); select ROIs in the list instead.")

In [ ]:
import getpass
import os

os.environ["DJ_IGNORE_CONFIG_FILE"] = "true"

import datajoint as dj

# 1. Connect to your schema

Same configuration as in your populate notebook. Replace the placeholders (marked `TODO`):
- the path of your DataJoint config file,
- `YOUR_PROJECT`: your project name as given to `make_project.py`. It defines the schema name
  (`ageuler_<username>_<project>`) and the schema module (`djimaging.user.<project>.schemas.<project>_schema`).

In [ ]:
username = getpass.getuser()
home_directory = os.path.expanduser("~")
username, home_directory

In [ ]:
# Set config file
config_file = f'{home_directory}/<path/to/your/datajoint/config>.json'  # TODO: your DataJoint config file
assert os.path.isfile(config_file), f'Set the path to your config file: {config_file}'

In [ ]:
# Define a schema name, by default made of your login name and the project name given to make_project.py
# It should start with ageuler and have some meaningful name after that
project = 'YOUR_PROJECT'  # TODO: your project name
assert project != 'YOUR_PROJECT', 'Set project to your project name (as given to make_project.py)'
schema_name = f"ageuler_{username}_{project}"

In [ ]:
# Load configuration for user
dj.config.load(config_file)
print("schema_name:", schema_name)
dj.conn()

## Create or load schema

In [ ]:
# Import your own schema, which defines the tables you will have in your database and how they are connected.
# TODO: replace YOUR_PROJECT with your project name (as above)
from djimaging.user.YOUR_PROJECT.schemas.YOUR_PROJECT_schema import *

## Activate schema

In [ ]:
from djimaging.utils.dj_utils import activate_schema

activate_schema(
    schema=schema,
    schema_name=schema_name,
    create_schema=True,
    create_tables=True,
)
schema

# 2. Choose a field and open CellViz

`CellViz.list_fields` lists the fields that have a ROI mask and populated `Traces`.
`CellViz.list_fields` and `CellViz.from_schema` take the `schema` object of your schema module (as here), the schema
module itself, or `globals()` after `from <your schema module> import *`.

In [ ]:
from djimaging.cellviz import CellViz

fields = CellViz.list_fields(schema)
print(f"{len(fields)} field(s) with traces")
fields

In [ ]:
# Any (partial) key that matches exactly one field of the table above, e.g. dict(date='2026-09-16', field='GCL0').
# None takes the first field. Pass raw_id=... / preprocess_id=... to `from_schema` if your schema has several.
#FIELD_KEY = None
FIELD_KEY = fields.iloc[0].to_dict()

In [ ]:
cv = CellViz.from_schema(schema, field_key=FIELD_KEY)
cv

## The viewer

- **Stimulus / Channel / Signal**: each presentation of the field (stimulus and conditions) is one entry; its own ROI
  mask (`RoiMask.RoiMaskPresentation`, i.e. shifted masks where the traces were extracted with them) is shown.
  Signal: raw `Traces` or `PreprocessTraces`.
- **Recording / HighRes**: the functional movie, or the static `HighRes` image with the ROIs registered onto it by
  image correlation (if the registration cannot be verified, the view stays at recording resolution and the reason
  is in *Details*).
- **Overlay** (both views): *ROI outlines* of the eligible ROIs, the *ROI mask* (filled labels; ROIs excluded by
  the quality filter in grey), the *ROI activity* (each ROI filled by its trace value at the current time) or *none*.
  Selected ROIs are always outlined in their colour.
- **All eligible ROIs / Selected ROIs**: heatmap and mean trace of the population, or the traces of the selected
  ROIs (click a cell with `%matplotlib widget`, or use the list).
- **QI filter** (chirp: `ChirpQI.qidx`, moving bar: `OsDsIndexes.d_qi`, off at start) directly above the ROI list.
  ROIs without a value are counted separately from ROIs below the threshold.
- **Trial mean ± SD**: the repeats are cut like `Snippets`; for the moving bar only the repeats of the current
  direction are summarised.
- **play / scrub / speed**: playback follows the recording clock (frames are skipped if rendering is too slow).
- **Export MP4** of the current view and time range, into the folder shown in the export row.

Keys, file paths and diagnostics are in the collapsed *Details* section.

# 3. Drive the viewer from code

Every control can also be set from code, e.g. to reproduce a view for an export. The figure above updates.

In [ ]:
[info.name for info in cv.source.list_presentations()]

In [ ]:
# ROI mask as overlay on the recording
cv.set_view('native')
cv.set_overlay('mask')

In [ ]:
# Select a few ROIs and show only their traces
cv.select_rois(cv.eligible[:3])
cv.set_display('selected')
cv.set_time(cv.pres.triggers[0] if cv.pres.triggers.size else 0.)  # jump to the first trigger

In [ ]:
# Quality filter (only for chirp and moving bar, if ChirpQI / OsDsIndexes are populated); None turns it off
if cv.pres.quality is not None:
    cv.set_threshold(0.35)
    print(cv.classes.summary())
else:
    print("No quality metric for this presentation:", cv.pres.quality_unavailable)

In [ ]:
# HighRes view with the ROI activity as overlay (falls back to the recording view if HighRes is not available)
cv.set_view('highres')
cv.set_overlay('activity')
print('view:', cv.view)

In [ ]:
# Other presentation / channel
# cv.select_presentation(cv.pres.info.name, channel='wDataCh0')

# 4. Export an MP4

The export renders every movie frame of the time range with the current view, overlay, selection and filter (a JSON
file with the same settings is written next to it). By default exports go to `<processed store>/cellviz/<schema>`;
change the folder in the export row or with `cv.export_dir = ...`. Writing into the `reference` store (the raw
data) is refused.

In [ ]:
print("Export folder:", cv.export_dir)

In [ ]:
from djimaging.cellviz.export import find_ffmpeg

EXPORT_MP4 = True  # writes a short video (5 s of recording) into the export folder

if EXPORT_MP4 and find_ffmpeg()[0] is not None:
    cv.set_view('native')
    cv.set_overlay('outlines')
    t0 = cv.current_time
    meta = cv.export(t_start=t0, t_stop=t0 + 5., speed=1.0)
    print(meta['file'], f"({meta['n_frames']} frames, {meta['fps']:.1f} fps)")
else:
    print("No export:", "EXPORT_MP4 = False" if not EXPORT_MP4 else find_ffmpeg()[1])

In [ ]:
if EXPORT_MP4 and cv.last_export is not None:
    from IPython.display import Video
    display(Video(cv.last_export['file'], embed=True, width=640))